# 03 - Prepare the ten-airport dataset

Prepare 2024 BTS flight records for ten selected departure airports

Keeps raw files unchanged, document exclusions, and retains seperate predictor, outcome, and identification fields

In [1]:
from pathlib import Path
import json

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "bts" / "2024"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SELECTION_FILE = (
    PROJECT_ROOT / "outputs" / "2024_audit" / "airport_selection.json"
)

assert RAW_DIR.is_dir(), f"Raw data folder not found: {RAW_DIR}"
assert SELECTION_FILE.is_file(), f"Selection file not found: {SELECTION_FILE}"

with SELECTION_FILE.open() as file:
    selection = json.load(file)

airports = selection["airports"]

assert len(airports) == 10
assert len(set(airports)) == 10

print("Project:", PROJECT_ROOT)
print("Selected airports:", ", ".join(airports))

Project: /Users/alainbouscholte/Developer/Thesis
Selected airports: ATL, DFW, DEN, ORD, CLT, LAX, PHX, LAS, SEA, LGA


In [2]:
predictors = ["Month", "DayOfWeek", "Reporting_Airline", "Dest", "CRSDepTime", "Distance"]
metadata = ["FlightDate", "Origin", "Flight_Number_Reporting_Airline"]
outcomes = ["DepDel15", "DepDelay", "Cancelled"]

columns = predictors + metadata + outcomes

Load each month, keep eligible flights from the 10 airports and records exclusions


In [3]:
frames, counts = [], []

for path in sorted(RAW_DIR.glob("otr_*_2024.csv")):
    df = pd.read_csv(path, usecols=columns, dtype={"CRSDepTime": "string"})
    selected = df[df["Origin"].isin(airports)]
    operated = selected[selected["Cancelled"].eq(0)]
    clean = operated.dropna(subset=["DepDel15"])

    frames.append(clean)
    counts.append({
        "file": path.name,
        "raw": len(df),
        "selected": len(selected),
        "cancelled": len(selected) - len(operated),
        "missing_label": len(operated) - len(clean),
        "retained": len(clean),
    })

flights = pd.concat(frames, ignore_index=True)
audit = pd.DataFrame(counts)
del frames

display(audit)
print(f"Retained {len(flights):,} flights")

,file,raw,selected,cancelled,missing_label,retained
0,otr_01_2024.csv,547271,179944,6345,0,173599
1,otr_02_2024.csv,519221,171373,825,0,170548
2,otr_03_2024.csv,591767,194036,1586,0,192450
3,otr_04_2024.csv,582185,194044,1240,0,192804
4,otr_05_2024.csv,609743,205501,3393,0,202108
5,otr_06_2024.csv,611132,204001,2332,0,201669
6,otr_07_2024.csv,634613,211208,6245,0,204963
7,otr_08_2024.csv,619025,208166,3339,0,204827
8,otr_09_2024.csv,582622,199172,1017,0,198155
9,otr_10_2024.csv,615497,208536,1032,0,207504


Retained 2,335,174 flights


Comparing cancellation rates

In [4]:
audit["cancelled_pct"] = 100 * audit["cancelled"] / audit["selected"]
display(audit[["file", "selected", "cancelled", "cancelled_pct"]].round(2))

,file,selected,cancelled,cancelled_pct
0,otr_01_2024.csv,179944,6345,3.53
1,otr_02_2024.csv,171373,825,0.48
2,otr_03_2024.csv,194036,1586,0.82
3,otr_04_2024.csv,194044,1240,0.64
4,otr_05_2024.csv,205501,3393,1.65
5,otr_06_2024.csv,204001,2332,1.14
6,otr_07_2024.csv,211208,6245,2.96
7,otr_08_2024.csv,208166,3339,1.60
8,otr_09_2024.csv,199172,1017,0.51
9,otr_10_2024.csv,208536,1032,0.49


Validate the combined dataset before saving

In [7]:
key = [
    "FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline",
    "Origin", "Dest", "CRSDepTime",
]

flights["FlightDate"] = pd.to_datetime(flights["FlightDate"])

assert len(audit) == 12
assert len(flights) == audit["retained"].sum()
assert set(flights["Origin"]) == set(airports)
assert flights["FlightDate"].dt.year.eq(2024).all()
assert flights["FlightDate"].dt.month.nunique() == 12
assert flights["Cancelled"].eq(0).all()
assert flights["DepDel15"].isin([0, 1]).all()
assert flights["DepDelay"].notna().all()
assert flights["DepDel15"].eq(flights["DepDelay"].ge(15)).all()
assert not flights.duplicated(key).any()

display(flights[columns].isna().sum().rename("missing"))
print(f"Checks passed: {len(flights):,} flights")

Month                              0
DayOfWeek                          0
Reporting_Airline                  0
Dest                               0
CRSDepTime                         0
Distance                           0
FlightDate                         0
Origin                             0
Flight_Number_Reporting_Airline    0
DepDel15                           0
DepDelay                           0
Cancelled                          0
Name: missing, dtype: int64

Checks passed: 2,335,174 flights


Make into compressed CSV files

In [8]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

flights.to_csv(
    PROCESSED_DIR / "flights_2024_top10.csv.gz",
    index=False,
)
audit.to_csv(
    PROJECT_ROOT / "outputs" / "2024_audit" / "cleaning_counts.csv",
    index=False,
)

print("Dataset and cleaning audit saved")

Dataset and cleaning audit saved
